In [2]:
import os

os.environ["JAVA_HOME"] = "/opt/homebrew/Cellar/openjdk@17/17.0.17/libexec/openjdk.jdk/Contents/Home"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [3]:
import subprocess

subprocess.run(["java", "-version"])

openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment Homebrew (build 17.0.17+0)
OpenJDK 64-Bit Server VM Homebrew (build 17.0.17+0, mixed mode, sharing)


CompletedProcess(args=['java', '-version'], returncode=0)

In [4]:
# importar bibliotecas
import pyspark
from pyspark.sql import SparkSession

In [5]:
#criando a sessão
sc = pyspark.SparkContext.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 14:52:52 WARN Utils: Your hostname, MacBook-Air-de-Leticia-2.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.106 instead (on interface en0)
26/06/08 14:52:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 14:52:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
#funcoes auxiliares

#parse_line: converte uma linha do csv em um dicionário, ou retorna None se a linha for inválida
def parse_line(line):

    fields = line.split(";")

    if len(fields) != 10:
        return None

    country, year_str, _, _, flow_str, price_str, weight_str, _, amount_str, category_str = fields

    try:
        year = int(year_str)
    except:
        return None

    try:
        price = float(price_str) if price_str else None
    except:
        return None

    try:
        weight = float(weight_str) if weight_str else None
    except:
        return None

    try:
        amount = float(amount_str) if amount_str else None
    except:
        return None

    return {
        
        #chaves do dicionario
        "country": country.strip(),
        "year": year,
        "flow": flow_str.strip().upper(),
        "price": price,
        "weight": weight,
        "amount": amount,
        "category": category_str.strip().upper()
    }


def save_txt(filename, lines):

    with open(filename, "w") as f:

        for line in lines:
            f.write(line + "\n")

In [7]:
#cada linha do csv vira um elemento do RDD
raw_rdd = sc.textFile("../in/operacoes_comerciais_inteira.csv")

In [8]:
#lendo o arquivo csv

#remove o cabeçalho do csv e converte cada linha em um dicionário
header = raw_rdd.first()

data_rdd = (
    raw_rdd
    .filter(lambda line: line != header)
    
    #transforma string em dicionario, retorna None se a linha for inválida
    .map(parse_line)
    .filter(lambda d: d is not None)
)

#antes: brazil;2016...
#depois: {"country": "brazil", "year": 2016, ...}

In [ ]:
#questao 1 (transações envolvendo brasil)
brazil_rdd = data_rdd.filter(
    lambda d: d["country"] == "Brazil"
)

#conta quantos elementos sobraram apos o filtro
total_brazil = brazil_rdd.count()

print("Questão 1:")
print(total_brazil)

save_txt(
    "resultado1.txt",
    [str(total_brazil)]
)

Questão 1:
184748


In [ ]:
#questao 2 (mesma contagem mas com pairRDD)

#funcao map transforma cada registro em um par (chave, valor) onde a chave é "Brazil" e o valor é 1
#funcao reduceByKey agrupa os valores pela chave, ou seja, conta quantos pares tem a chave "Brazil"
brazil_pair = brazil_rdd \
    .map(lambda d: ("Brazil", 1)) \
    .reduceByKey(lambda a, b: a + b)

key2, val2 = brazil_pair.collect()[0]

print("Questão 2:")
print(key2, val2)

save_txt(
    "resultado2.txt",
    [f"{key2}\t{val2}"]
)

Questão 2:
Brazil 184748


In [ ]:
#questao 3 (transacoes brasil 2016)
brazil_2016 = brazil_rdd.filter(
    
    #filtra os registros que tem o ano igual a 2016
    lambda d: d["year"] == 2016
)

#soma o que sobrou apos o filtro
total_brazil_2016 = brazil_2016.count()

print("Questão 3:")
print(total_brazil_2016)

save_txt(
    "resultado3.txt",
    [str(total_brazil_2016)]
)


Questão 3:
6210


In [ ]:
#questao 4 (brasil 2016 mas com pairRDD)

#funcao map transforma cada registro em um par (chave, valor) onde a chave é (Brazil, 2016) e o valor é 1, colocamos 1 porque queremos contar quantos existem assim
#funcao reduceByKey agrupa os valores pela chave, ou seja, conta quantos pares tem a chave (Brazil, 2016)
brazil2016_pair = brazil_2016 \
    .map(lambda d: ((d["country"], d["year"]), 1)) \
    .reduceByKey(lambda a, b: a + b)

#transforma o resultado em lista python e quebra a lista para pegar a chave e o valor, como só tem um par, pegamos o primeiro elemento da lista
#c4 = Brazil, y4 = 2016, v4 = total_brazil_2016
(c4, y4), v4 = brazil2016_pair.collect()[0]

print("Questão 4:")
print(c4, y4, v4)

save_txt(
    "resultado4.txt",
    [f"{c4},{y4}\t{v4}"]
)


Questão 4:
Brazil 2016 6210


In [ ]:
#questao 5 Número de transações por flow e por ano, ordenados por ano a partir de 2010, transformando Flow em maiúsculas. Chave = (ano, flow) e valor.

resultado05 = (
    data_rdd
    .filter(lambda d: d["year"] >= 2010)
    
    #criando a chave (ano, flow) e o valor 1 para contar quantos existem de cada chave, transformando flow em maiusculas
    .map(
        lambda d: (
            (d["year"], d["flow"].upper()),
            1
        )
    )
    #soma os valores de cada chave, ou seja, conta quantos pares tem cada chave (ano, flow)
    .reduceByKey(lambda a, b: a + b)
    #sorta o resultado por ano, que é a primeira parte da chave
    .sortBy(lambda x: x[0][0])
    
    #resultado vira uma lista de tuplas, onde cada tupla é ((ano, flow), count)
    .collect()
)

print("Questão 5:")

for item in resultado05:
    print(item)

save_txt(
    "resultado05.txt",
    [str(x) for x in resultado05]
)

Questão 5:
((2010, 'RE-IMPORT'), 9273)
((2010, 'RE-EXPORT'), 17058)
((2010, 'IMPORT'), 222447)
((2010, 'EXPORT'), 125013)
((2011, 'RE-IMPORT'), 10273)
((2011, 'RE-EXPORT'), 17564)
((2011, 'EXPORT'), 125909)
((2011, 'IMPORT'), 220850)
((2012, 'RE-IMPORT'), 10337)
((2012, 'EXPORT'), 128072)
((2012, 'IMPORT'), 222034)
((2012, 'RE-EXPORT'), 16900)
((2013, 'IMPORT'), 216282)
((2013, 'EXPORT'), 127904)
((2013, 'RE-IMPORT'), 9992)
((2013, 'RE-EXPORT'), 16758)
((2014, 'RE-EXPORT'), 20139)
((2014, 'RE-IMPORT'), 10424)
((2014, 'EXPORT'), 125142)
((2014, 'IMPORT'), 208749)
((2015, 'RE-IMPORT'), 10288)
((2015, 'RE-EXPORT'), 18966)
((2015, 'IMPORT'), 203920)
((2015, 'EXPORT'), 125878)
((2016, 'EXPORT'), 106104)
((2016, 'RE-IMPORT'), 7024)
((2016, 'IMPORT'), 160235)
((2016, 'RE-EXPORT'), 16147)


In [ ]:
#questao 6 Média da coluna Price para o ano de 2016 utilizando RDD.

price_2016 = (
    data_rdd
    
    #filtrando onde o ano é 2016 e o preço não é None
    .filter(
        lambda d:
            d["year"] == 2016
            and d["price"] is not None
    )
    
    #pegando apenas os preços para calcular a média, sem o ano mais (MAP É UMA TRANSFORMATION)
    .map(lambda d: d["price"])
)

soma = price_2016.sum()

#count serve par contar quantos elementos o RDD tem
quantidade = price_2016.count()

media = soma / quantidade

print("Questão 6:")
print(media)

save_txt(
    "resultado06.txt",
    [str(media)]
)

Questão 6:
137984606.1966806


In [ ]:
# Média de preço no Brasil em 2016

#criando um RDD filtrando apenas os registros do Brasil em 2016 e onde o preço não é None
brasil_2016 = data_rdd.filter(
    lambda d: d["country"] == "Brazil" and d["year"] == 2016 and d["price"] is not None
)

#pair rdd onde a chave é "Brasil-2016" e o valor é uma tupla (preço, 1), onde preço é o valor da coluna price e 1 é para contar quantos registros tem
#guardamos (preco,1) porque precisamos da soma deles e da quantidade de registros
pair_rdd = brasil_2016.map(
    lambda d:
    (
        "Brasil-2016",
        (
            d["price"],
            1
        )
    )
)

#reduceByKey para somar os preços e contar quantos registros tem, ou seja, a chave "Brasil-2016" vai ter o valor (soma dos preços, quantidade de registros)
#com essa funcao lambda somamos separadmaente os preços e as quantidades
resultado07 = pair_rdd.reduceByKey(
    lambda a, b:
    (
        a[0] + b[0],
        a[1] + b[1]
    )
    
    #collect para transformar o pair rdd em lista
).collect()

print("Questão 7:")

for chave, valores in resultado07:

    soma_precos = valores[0]
    quantidade = valores[1]

    media = soma_precos / quantidade

    print("Média:", media)

save_txt(
    "resultado07.txt",
    [
        f"Média de preços -> {media}"
    ]
)

Questão 7:
Média: 92889190.54396135


In [ ]:
#questao 8 O preço máximo e mínimo por categoria e por ano, ordenado por país, sendo que a coluna categoria deve conter valores com letra maiúsculas. 


resultado08 = (
    data_rdd

    # remove registros sem preço
    .filter(lambda d: d["price"] is not None)

    # cria chave (ano, categoria)
    #chave é uma tupla (ano, categoria) e o valor é uma tupla (preço, preço), onde o primeiro preço é para calcular o máximo e o segundo preço é para calcular o mínimo
    #valor no começo é o preço 2x porque para calcular o máximo e o mínimo precisamos comparar os preços, ou seja, o preço do registro atual com o preço máximo e mínimo encontrados até agora
    .map(
        lambda d: (
            (
                d["year"],
                d["category"].upper()
            ),
            (
                d["price"],
                d["price"]
            )
        )
    )

    # calcula máximo e mínimo
    #agrupa por chaves iguais
    #vai fazendo um loop de comparação entre os preços para encontrar o máximo e o mínimo, ou seja, compara o preço do registro atual com o preço máximo e mínimo encontrados até agora
    .reduceByKey(
        lambda a, b: (
            max(a[0], b[0]),
            min(a[1], b[1])
        )
    )

    # ordena por ano
    .sortBy(lambda x: x[0][0])

    .collect()
)

print("Questão 8:")

linhas_saida = []

for chave, valores in resultado08:

    ano = chave[0]
    categoria = chave[1]

    maximo = valores[0]
    minimo = valores[1]

    texto = (
        f"Ano: {ano} | "
        f"Categoria: {categoria} | "
        f"Máximo: {maximo} | "
        f"Mínimo: {minimo}"
    )

    print(texto)

    linhas_saida.append(texto)

save_txt(
    "resultado08.txt",
    linhas_saida
)

NameError: name 'data_rdd' is not defined

In [ ]:
#questao 9  país com a maior exportação

export_rdd = data_rdd.filter(
    lambda d: d["flow"] == "EXPORT"
)

export_por_pais = export_rdd \
    .map(lambda d: ((d["country"], "EXPORT"), 1)) \
    .reduceByKey(lambda a, b: a + b)

#retorna somente 1 valor
# x é cada elemento do rdd, e [1] é o valor (quantidade de exportacoes)
#takeordered é mais eficaz que sortBy para pegar os maiores ou menores valores, porque ele não precisa ordenar todo o RDD, apenas os n primeiros, onde n é o número de elementos que queremos pegar (no caso, 1)
#por causa do -, ele vai pegar o maior valor, porque o takeOrdered pega os menores valores, então ao colocar o valor negativo, ele inverte a ordem e pega o maior valor
resultado09 = export_por_pais.takeOrdered(
    1,
    key=lambda x: -x[1]
)

print("Questão 9:")

for item in resultado09:
    print(item)

save_txt(
    "resultado09.txt",
    [str(x) for x in resultado09]
)

#saida = ((chave (pais, flow), valor (quantidade de transações)),)

Questão 9:
(('Australia', 'EXPORT'), 119815)


In [ ]:
#questao 10 preço mínimo por país e por ano, filtrado de forma crescente por ano e país

resultado10 = data_rdd \
    .filter(lambda d: d["price"] is not None) \
    .map(
        lambda d: (
            (d["year"], d["country"]),
            d["price"]
        )
    ) \
    .reduceByKey(min) \
    .sortBy(lambda x: (x[0][0], x[0][1])) \
    .collect()

#reduceByKey(min) vai comparar os preços de cada país e ano e retornar o menor preço encontrado para cada chave (ano, país)
print("Questão 10:")

for item in resultado10:
    print(item)

save_txt(
    "resultado10.txt",
    [str(x) for x in resultado10]
)

Questão 10:
((1988, 'Australia'), 7.0)
((1988, 'Finland'), 2.0)
((1988, 'Fmr Fed. Rep. of Germany'), 1000.0)
((1988, 'Greece'), 2.0)
((1988, 'Haiti'), 101.0)
((1988, 'Iceland'), 8.0)
((1988, 'India'), 6.0)
((1988, 'Japan'), 1607.0)
((1988, 'Portugal'), 5.0)
((1988, 'Rep. of Korea'), 1.0)
((1988, 'Switzerland'), 7.0)
((1988, 'Thailand'), 4.0)
((1989, 'Australia'), 1.0)
((1989, 'Bangladesh'), 1.0)
((1989, 'Brazil'), 1.0)
((1989, 'Canada'), 4.0)
((1989, 'Cyprus'), 2.0)
((1989, 'Denmark'), 95.0)
((1989, 'Finland'), 7.0)
((1989, 'Fmr Fed. Rep. of Germany'), 1000.0)
((1989, 'Greece'), 7.0)
((1989, 'Haiti'), 1171.0)
((1989, 'Iceland'), 3.0)
((1989, 'India'), 30.0)
((1989, 'Indonesia'), 3.0)
((1989, 'Japan'), 1527.0)
((1989, 'Malaysia'), 4.0)
((1989, 'New Zealand'), 5.0)
((1989, 'Oman'), 119.0)
((1989, 'Paraguay'), 2.0)
((1989, 'Portugal'), 4.0)
((1989, 'Rep. of Korea'), 1.0)
((1989, 'Romania'), 3000.0)
((1989, 'Singapore'), 6.0)
((1989, 'Spain'), 4.0)
((1989, 'Switzerland'), 1.0)
((1989, 'Tha

In [ ]:
#questao 11 Transação que teve o maior preço por kg do tipo export.

export_rdd = data_rdd.filter(
    lambda d:
        d["flow"] == "EXPORT"
        and d["price"] is not None
        and d["weight"] is not None
        and d["weight"] > 0
)

#chave é o preço por kg, e o valor uma tupla

preco_por_kg = export_rdd.map(
    lambda d: (
        d["price"] / d["weight"],
        (
            d["year"],
            d["country"],
            d["category"]
        )
    )
)

#takeordered vai retornar 1 valor, x é cada elemento do rdd, e [0] é o valor (preço por kg)
#ordena os preços por kg de forma decrescente e pega o primeiro, ou seja, o maior preço por kg
resultado11 = preco_por_kg.takeOrdered(
    1,
    key=lambda x: -x[0]
)

print("Questão 11:")

for valor, info in resultado11:

    ano, pais, categoria = info

    print("Preço por kg:", valor)
    print("Ano:", ano)
    print("País:", pais)
    print("Categoria:", categoria)

save_txt(
    "resultado11.txt",
    [
        f"Preço por kg: {resultado11[0][0]}",
        f"Ano: {resultado11[0][1][0]}",
        f"País: {resultado11[0][1][1]}",
        f"Categoria: {resultado11[0][1][2]}"
    ]
)

Questão 11:
Preço por kg: 292668800.0
Ano: 1999
País: Norway
Categoria: 89_SHIPS_BOATS_AND_OTHER_FLOATING_STRUCTURES
